# Notebook 5: Movie Chatbot with RAG (PDF Documents)

## Features:
- ✅ PDF document search (Oscars 2026)
- ✅ Web search for current information
- ✅ Conversational memory
- ✅ Semantic document retrieval
- ✅ Two-tool agentic system

## What is RAG?
**Retrieval-Augmented Generation** - Combines document search with LLM generation to provide accurate, grounded answers from your documents.

## Use Case:
Search the 2026 Academy Awards nominations document and get accurate information about nominees, categories, and winners.

## Step 1: Installation

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results pypdf faiss-cpu --quiet

print("✓ Packages installed")

## Step 2: Imports

In [12]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from ai_course_utils import (
    load_llm_from_env,
    load_use_case_config,
    load_pdf_for_rag,
    display_config
)

# Load environment
load_dotenv()
load_dotenv('../.env')

print("✓ Imports successful")

✓ Imports successful


## Step 3: Configuration

In [13]:
display_config()

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')


## Step 4: Load Use Case Configuration

In [14]:
# Load use case
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"
use_case_config = load_use_case_config(use_case_file)
system_prompt = use_case_config.get("agent_prompt", "You are a helpful movie assistant")

print("✓ System prompt loaded")

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url
✓ System prompt loaded


## Step 5: Load PDF Document

**USER INPUT**: Specify the path to your PDF file.

In [15]:
# ============================================================================
# USER INPUT: Specify PDF path
# ============================================================================
pdf_file = "../data/The_98th_Academy_Awards_2026.pdf"

print(f"Loading PDF: {pdf_file}")
print("This may take a moment...")

# Load and process PDF
documents, vectorstore, retriever = load_pdf_for_rag(
    filepath=pdf_file,
    chunk_size=1000,
    chunk_overlap=200
)

print(f"\n✅ PDF loaded successfully!")
print(f"  Pages: {len(documents)}")
print(f"  Ready for semantic search")

Loading PDF: ../data/The_98th_Academy_Awards_2026.pdf
This may take a moment...
📄 Loading PDF: ../data/The_98th_Academy_Awards_2026.pdf
  ✓ Loaded 11 pages
  ✓ Created 14 chunks
  ✓ Vector store created
  ✓ Retriever ready (top 3 results)

✅ PDF loaded successfully!
  Pages: 11
  Ready for semantic search


## Step 6: Test the Retriever

Test that the PDF was loaded correctly and retriever works.

In [16]:
# Test retrieval directly
test_query = "Who is nominated for Best Picture?"
print(f"Test Query: {test_query}\n")

results = retriever.invoke(test_query)
print(f"Retrieved {len(results)} relevant chunks:")
for i, doc in enumerate(results, 1):
    print(f"\nChunk {i}:")
    print(doc.page_content[:200] + "...")
    
print("\n✓ Retriever working correctly")

Test Query: Who is nominated for Best Picture?

Retrieved 3 relevant chunks:

Chunk 1:
Nominees to be determined
THE PERFECT NEIGHBOR
Geeta Gandbhir, Alisa Payne, Nikon Kwantu and Sam Bisbee
DOCUMENTARY SHORT FILM
NOMINEES
ALL THE EMPTY ROOMS
Joshua Seftel and Conall Jones
ARMED ONLY WI...

Chunk 2:
COSTUME DESIGN
NOMINEES
AVATAR: FIRE AND ASH
Deborah L. Scott
FRANKENSTEIN
Kate Hawley
HAMNET
Malgosia Turzanska
MARTY SUPREME
Miyako Bellizzi
SINNERS
Ruth E. Carter
DIRECTING
NOMINEES
HAMNET
Chloé Zh...

Chunk 3:
VISUAL EFFECTS
NOMINEES
AVATAR: FIRE AND ASH
Joe Letteri, Richard Baneham, Eric Saindon and Daniel Barrett
F1
Ryan Tudhope, Nicolas Chevallier, Robert Harrington and Keith Dawson
JURASSIC WORLD REBIRT...

✓ Retriever working correctly


## Step 7: Define Tools

Define both search_web and search_documents tools.

In [17]:
# Tool 1: Web Search
@tool
def search_web(query: str) -> str:
    """
    Search the web for current movie information and news.
    
    Use for:
    - Current box office information
    - Recent movie news
    - General movie queries
    - Information not in the PDF document
    
    Args:
        query: Search query
        
    Returns:
        Web search results
    """
    try:
        search = GoogleSerperAPIWrapper()
        return search.run(query)
    except Exception as e:
        return f"Search error: {str(e)}"

print("✓ Tool 1: search_web defined")

✓ Tool 1: search_web defined


In [18]:
# Tool 2: Document Search (RAG)
@tool
def search_documents(query: str) -> str:
    """
    Search the loaded PDF document (2026 Academy Awards nominations).
    
    Use for:
    - Questions about Oscar nominations
    - Information about nominees and categories
    - Details from the awards document
    - Specific facts in the PDF
    
    Args:
        query: Search query
        
    Returns:
        Relevant content from the PDF document
    """
    try:
        docs = retriever.invoke(query)
        
        results = []
        for i, doc in enumerate(docs, 1):
            results.append(f"[Excerpt {i}]\n{doc.page_content}")
        
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Error searching documents: {str(e)}"

print("✓ Tool 2: search_documents defined")

✓ Tool 2: search_documents defined


In [19]:
# Configure tools list
tools = [search_web, search_documents]

print(f"\n✅ Tools configured: {[t.name for t in tools]}")


✅ Tools configured: ['search_web', 'search_documents']


## Step 8: Initialize LLM with Tools

**IMPORTANT**: This must come AFTER tool definitions.

In [20]:
# Load LLM and bind tools
llm = load_llm_from_env()
model = llm.bind_tools(tools)

print("✓ LLM initialized")
print("✓ Tools bound to model")

🤖 Loading LLM: openai / gpt-4.1-mini
✓ LLM initialized
✓ Tools bound to model


## Step 9: Build Agent Graph with Memory

Now we can build the agent graph using the model.

In [21]:
# Define agent behavior
def call_model(state: MessagesState):
    """Agent node - decides which tools to use."""
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState):
    """Decision function - continue to tools or end."""
    last_message = state["messages"][-1]
    return "tools" if last_message.tool_calls else END

# Build graph
workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))
workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END}
)
workflow.add_edge("tools", "agent")

# Compile with memory
app = workflow.compile(checkpointer=MemorySaver())

print("✓ Agent graph compiled")
print("✓ Memory enabled")

✓ Agent graph compiled
✓ Memory enabled


## Step 10: Chat Helper Function

In [22]:
def chat(user_input: str, thread_id: str = "default", verbose: bool = False) -> str:
    """
    Chat with the RAG-enabled assistant.
    
    Args:
        user_input: Your question
        thread_id: Conversation thread ID
        verbose: Show which tools are used
        
    Returns:
        Assistant's response
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"Query: {user_input[:100]}...")
        print(f"{'='*70}")
        tools_used = []
    
    result = None
    for event in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config,
        stream_mode="values"
    ):
        result = event
        if verbose:
            last_message = event["messages"][-1]
            if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
                for tool_call in last_message.tool_calls:
                    tool_name = tool_call['name']
                    if tool_name not in tools_used:
                        tools_used.append(tool_name)
                        print(f"  🔧 Using: {tool_name}")
    
    if verbose and tools_used:
        print(f"  ✓ Tools: {', '.join(tools_used)}")
    
    return result["messages"][-1].content

print("\n🎬 RAG-Enabled Movie Chatbot Ready!")
print("\nUsage:")
print("  response = chat('Your question')")
print("  response = chat('Follow-up', thread_id='same')")
print("  response = chat('Question', verbose=True)  # See tools")


🎬 RAG-Enabled Movie Chatbot Ready!

Usage:
  response = chat('Your question')
  response = chat('Follow-up', thread_id='same')
  response = chat('Question', verbose=True)  # See tools


## Step 11: Test Document Search

Test queries about the Oscars PDF.

In [23]:
# Test 1: Best Picture nominations
print("="*70)
print("Test 1: Best Picture Nominations")
print("="*70)

query1 = "Who is nominated for Best Picture?"
print(f"\nUser: {query1}")
response1 = chat(query1, thread_id="test1", verbose=True)
print(f"\nBot: {response1}")

Test 1: Best Picture Nominations

User: Who is nominated for Best Picture?

Query: Who is nominated for Best Picture?...
  🔧 Using: search_documents
  ✓ Tools: search_documents

Bot: The document does not list the nominees for Best Picture specifically. Would you like me to search the web for the current Best Picture nominations for the 2026 Academy Awards?


In [24]:
# Test 2: Specific nominee
print("\n" + "="*70)
print("Test 2: Specific Nominee Details")
print("="*70)

query2 = "Tell me about the movie 'Marty Supreme'"
print(f"\nUser: {query2}")
response2 = chat(query2, thread_id="test2", verbose=True)
print(f"\nBot: {response2}")


Test 2: Specific Nominee Details

User: Tell me about the movie 'Marty Supreme'

Query: Tell me about the movie 'Marty Supreme'...
  🔧 Using: search_web
  ✓ Tools: search_web

Bot: "Marty Supreme" is a 2025 American sports comedy-drama film directed by Josh Safdie. Set in the 1950s, it follows Marty Mauser, a 23-year-old Jewish man from a Jewish ghetto who is a grifter, hustler, salesman, and one of the world's best ping-pong players. The story is about his pursuit of greatness despite the lack of respect for his dream. The film features a strong ensemble cast including Timothée Chalamet, Gwyneth Paltrow, Odessa A'zion, and Tyler the Creator. It has been praised for its storytelling, acting, and score, described as an explosive and unpredictable cinematic experience. The release date is December 25, 2025. Would you like to know more about the cast or the director?


## Step 12: Test Conversational Memory

Test that the chatbot remembers previous context.

In [25]:
# Test conversation with memory
thread = "memory_test"

print("\n" + "="*70)
print("Test: Conversational Memory")
print("="*70)

# Query 1
print("\n[Message 1]")
query_a = "Who is nominated for Best Actor?"
print(f"User: {query_a}")
response_a = chat(query_a, thread_id=thread, verbose=True)
print(f"\nBot: {response_a[:250]}...")

# Query 2 (should remember context)
print("\n" + "-"*70)
print("\n[Message 2 - Testing Memory]")
query_b = "Tell me more about the first one you mentioned"
print(f"User: {query_b}")
response_b = chat(query_b, thread_id=thread, verbose=True)
print(f"\nBot: {response_b[:250]}...")

# Query 3 (still remembers)
print("\n" + "-"*70)
print("\n[Message 3 - Still Testing Memory]")
query_c = "What movie is that nominee in?"
print(f"User: {query_c}")
response_c = chat(query_c, thread_id=thread, verbose=True)
print(f"\nBot: {response_c[:250]}...")


Test: Conversational Memory

[Message 1]
User: Who is nominated for Best Actor?

Query: Who is nominated for Best Actor?...
  🔧 Using: search_documents
  ✓ Tools: search_documents

Bot: The nominees for Best Actor (Actor in a Leading Role) at the 2026 Academy Awards are:

- Timothée Chalamet for "Marty Supreme"
- Leonardo DiCaprio for "One Battle after Another"
- Ethan Hawke for "Blue Moon"
- Michael B. Jordan for "Sinners"
- Wagner...

----------------------------------------------------------------------

[Message 2 - Testing Memory]
User: Tell me more about the first one you mentioned

Query: Tell me more about the first one you mentioned...

Bot: Timothée Chalamet is nominated for Best Actor for his role in the film "Marty Supreme." If you'd like, I can provide more details about the movie "Marty Supreme," such as its plot, genre, or other nominations it may have received. Would you like me t...

----------------------------------------------------------------------

[Message 3 - 

## Step 13: Test Bollywood Query

Test the specific query about Bollywood movies.

In [29]:
# Test Bollywood awards query
thread = "bollywood_test"

print("\n" + "="*70)
print("Test: Bollywood Movie Query")
print("="*70)

query = "Is there any Bollywood movie receiving award in this award document just loaded?"
print(f"\nUser: {query}")
response = chat(query, thread_id=thread, verbose=True)
print(f"\nBot: {response}")

query2 = "Which movie were nominated for Best Actress and who is the best actress?"
print(f"\nUser: {query2}")
response = chat(query2, thread_id=thread, verbose=True)
print(f"\nBot: {response}")


Test: Bollywood Movie Query

User: Is there any Bollywood movie receiving award in this award document just loaded?

Query: Is there any Bollywood movie receiving award in this award document just loaded?...

Bot: I have checked the 2026 Academy Awards document, and there is no Bollywood movie listed as receiving an award or nomination. If you want, I can help you find Bollywood movies that have been recognized in other recent awards or festivals. Would you like that?

User: Which movie were nominated for Best Actress and who is the best actress?

Query: Which movie were nominated for Best Actress and who is the best actress?...
  🔧 Using: search_documents
  ✓ Tools: search_documents

Bot: The nominees for Best Actress (Actress in a Leading Role) in the 2026 Academy Awards are:

- Jessie Buckley for "Hamnet"
- Rose Byrne

The document does not specify the winner of the Best Actress award. Would you like me to check for the winner from another source?


## Step 14: Test Web Search vs Document Search

Compare how different types of queries are handled.

In [27]:
# Test that shows difference between document search and web search
print("\n" + "="*70)
print("Test: Web Search vs Document Search")
print("="*70)

# Query about PDF content (should use search_documents)
query_pdf = "What are the Best Director nominations from the PDF?"
print(f"\n[PDF Query]")
print(f"User: {query_pdf}")
response_pdf = chat(query_pdf, thread_id="compare1", verbose=True)
print(f"\nBot: {response_pdf[:200]}...")

print("\n" + "-"*70)

# Query about current info (should use search_web)
query_web = "What are the current top-grossing movies of 2026?"
print(f"\n[Web Query]")
print(f"User: {query_web}")
response_web = chat(query_web, thread_id="compare2", verbose=True)
print(f"\nBot: {response_web[:200]}...")


Test: Web Search vs Document Search

[PDF Query]
User: What are the Best Director nominations from the PDF?

Query: What are the Best Director nominations from the PDF?...
  🔧 Using: search_documents
  ✓ Tools: search_documents

Bot: The Best Director nominations from the PDF are:

- Chloé Zhao for "Hamnet"
- Josh Safdie for "Marty Supreme"
- Paul Thomas Anderson for "One Battle After Another"
- Joachim Trier for "Sentimental Valu...

----------------------------------------------------------------------

[Web Query]
User: What are the current top-grossing movies of 2026?

Query: What are the current top-grossing movies of 2026?...
  🔧 Using: search_web
  ✓ Tools: search_web

Bot: The current top-grossing movies of 2026 include:

1. Cheburashka 2 with $79,500,909
2. 28 Years Later: The Bone Temple with $53,775,943
3. Primate with $24,260,958
4. Mercy with $19,240,756
5. Send He...


## Step 15: Interactive Testing

Try your own queries!

In [28]:
# YOUR TURN: Ask about the Oscars document
my_query = "Who won Best Actress?"

print(f"User: {my_query}")
response = chat(my_query, thread_id="my_session", verbose=True)
print(f"\nBot: {response}")

User: Who won Best Actress?

Query: Who won Best Actress?...
  🔧 Using: search_documents
  ✓ Tools: search_documents

Bot: The document does not specify the winner for Best Actress, only the nominees. Would you like me to search the web for the current Best Actress winner?


## Summary

### ✅ What We Built:
- **PDF document search (RAG)** - Semantic search through Oscars 2026 document
- **Web search** - Current information and general queries
- **Conversational memory** - Remembers context across messages
- **Smart tool selection** - Agent decides which tool to use
- **Two-tool system** - search_web + search_documents

### 🎯 How RAG Works:
1. **Load PDF** → Split into chunks
2. **Create embeddings** → Convert text to vectors
3. **Build vector store** → FAISS database
4. **Search** → Find relevant chunks by similarity
5. **Generate** → LLM uses retrieved content to answer

### 🔧 Tools Available:
| Tool | Purpose | Example Query |
|------|---------|---------------|
| **search_documents** | Search Oscars PDF | "Who's nominated for Best Picture?" |
| **search_web** | Current movie info | "Top movies in theaters now" |

### 📊 Key Benefits:
- ✅ **Accurate answers** from your documents
- ✅ **Cited sources** - knows where info came from
- ✅ **No hallucinations** - grounded in actual content
- ✅ **Semantic search** - finds by meaning, not just keywords
- ✅ **Memory enabled** - maintains conversation context

### 💡 Usage Tips:
- Use **search_documents** for questions about the Oscars PDF
- Use **search_web** for current movie information
- Use same **thread_id** to continue conversations
- Use **verbose=True** to see which tools are being used

### 🚀 Next Steps:
- **Notebook 6**: Add TAG (genre taxonomy classification)
- **Notebook 7**: Combine RAG + TAG + all tools

### 🎓 Experiments to Try:
1. Ask about different award categories
2. Compare nominees across categories
3. Test follow-up questions (memory)
4. Try queries that need both tools
5. Test with your own PDF documents